In [0]:
from pyspark.sql import functions as F

In [0]:
product_df = (
    spark.read.
    format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load("/Volumes/business_to_business/sports_bar_data/products/*")
    .withColumn("ingestion_time",F.current_timestamp())
    .select("*", "_metadata.file_name","_metadata.file_size")

)

In [0]:
product_df.show(truncate=False)

+-------------------------------------------------+----------+-----------------+--------------------------+------------+---------+
|product_name                                     |product_id|category         |ingestion_time            |file_name   |file_size|
+-------------------------------------------------+----------+-----------------+--------------------------+------------+---------+
|SportsBar Energy Bar Choco Fudge (60g)           |25891101  |energy bars      |2026-09-05 06:34:55.600576|products.csv|1388     |
|SportsBar Energy Bar Choco Fudge (40g)           |25891102  |energy bars      |2026-09-05 06:34:55.600576|products.csv|1388     |
|SportsBar Energy Bar Choco Fudge (25g)           |25891103  |energy bars      |2026-09-05 06:34:55.600576|products.csv|1388     |
|SportsBar Protien Bar Peanut Crunch (45g)        |25891201  |protien bars     |2026-09-05 06:34:55.600576|products.csv|1388     |
|SportsBar Protien Bar Peanut Crunch (55g)        |25891202  |protien bars     |202

In [0]:
product_df.printSchema()

root
 |-- product_name: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



In [0]:
display(product_df)

product_name,product_id,category,ingestion_time,file_name,file_size
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars,2026-09-05T06:34:57.430Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars,2026-09-05T06:34:57.430Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars,2026-09-05T06:34:57.430Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars,2026-09-05T06:34:57.430Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars,2026-09-05T06:34:57.430Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars,2026-09-05T06:34:57.430Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals,2026-09-05T06:34:57.430Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals,2026-09-05T06:34:57.430Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals,2026-09-05T06:34:57.430Z,products.csv,1388
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy,2026-09-05T06:34:57.430Z,products.csv,1388


In [0]:
product_df.write\
.mode("overwrite")\
.format("delta")\
.option("delta.enableChangeDataFeed", "true")\
.saveAsTable("business_to_business.bronze.products")


In [0]:
product_df = (
    spark.read.table("business_to_business.bronze.products")
)

In [0]:
display(product_df)

product_name,product_id,category,ingestion_time,file_name,file_size
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars,2026-09-05T06:35:08.056Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars,2026-09-05T06:35:08.056Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars,2026-09-05T06:35:08.056Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars,2026-09-05T06:35:08.056Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars,2026-09-05T06:35:08.056Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars,2026-09-05T06:35:08.056Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals,2026-09-05T06:35:08.056Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals,2026-09-05T06:35:08.056Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals,2026-09-05T06:35:08.056Z,products.csv,1388
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy,2026-09-05T06:35:08.056Z,products.csv,1388


In [0]:
product_df.groupBy("product_id")\
          .agg(F.count("product_id").alias("count"))\
          .filter(F.col("count") >= 2)\
          .show()

+----------+-----+
|product_id|count|
+----------+-----+
|  25891101|    2|
|  25891102|    2|
+----------+-----+



In [0]:
product_df.count()

20

In [0]:
product_df = product_df.dropDuplicates(["product_id"])

In [0]:
product_df.count()

18

In [0]:
product_df.printSchema()

root
 |-- product_name: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)



In [0]:
product_df = (
    product_df.withColumn("category",
                          F.when(F.col("category").isNull(), None)
                          .otherwise(F.trim(F.col("category")))
                          )
)

In [0]:
product_df = product_df.withColumn("category", F.initcap(F.col("category")))

In [0]:
product_df.select(["product_name","category"]).show(truncate=False)

+-------------------------------------------------+-----------------+
|product_name                                     |category         |
+-------------------------------------------------+-----------------+
|SportsBar Energy Bar Choco Fudge (60g)           |Energy Bars      |
|SportsBar Energy Bar Choco Fudge (40g)           |Energy Bars      |
|SportsBar Energy Bar Choco Fudge (25g)           |Energy Bars      |
|SportsBar Protien Bar Peanut Crunch (45g)        |Protien Bars     |
|SportsBar Protien Bar Peanut Crunch (55g)        |Protien Bars     |
|SportsBar Protien Bar Peanut Crunch (65g)        |Protien Bars     |
|SportsBar Granola Crunch Honey Almond (400g)     |Granola & Cereals|
|SportsBar Granola Crunch Honey Almond (300g)     |Granola & Cereals|
|SportsBar Granola Crunch Honey Almond (200g)     |Granola & Cereals|
|SportsBar Greek Yogurt Pro Vanilla (200g)        |Recovery Dairy   |
|SportsBar Greek Yogurt Pro Vanilla (120g)        |Recovery Dairy   |
|SportsBar Greek Yog

In [0]:
product_df = (
    product_df.withColumn("product_name",
                          F.regexp_replace(F.col("product_name"), "(?i)Protien", "Protein")
    )
    .withColumn("category",
                F.regexp_replace(F.col("category"), "(?i)Protien", "Protein")
                )
)

In [0]:
product_df.printSchema()

root
 |-- product_name: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)



#### Standardizing the columns according to parent's fields

In [0]:

product_df = (
                product_df.withColumn("division",
                F.when(F.col("category") == "Energy Bars", "Nutrition Bars")
                .when(F.col("category") == "Protein Bars",       "Nutrition Bars")
                .when(F.col("category") == "Granola & Cereals",  "Breakfast Foods")
                .when(F.col("category") == "Recovery Dairy",     "Dairy & Recovery")
                .when(F.col("category") == "Healthy Snacks",     "Healthy Snacks")
                .when(F.col("category") == "Electrolyte Mix",    "Hydration & Electrolytes")
                .otherwise("Other")
                )
)

In [0]:
product_df.filter(F.col("division") == "Other").show()


+------------+----------+--------+--------------+---------+---------+--------+
|product_name|product_id|category|ingestion_time|file_name|file_size|division|
+------------+----------+--------+--------------+---------+---------+--------+
+------------+----------+--------+--------------+---------+---------+--------+



In [0]:
#separate the varient from  product name
product_df = product_df.withColumn("varient",
                      (
                          F.regexp_extract(F.col("product_name"), r"\((.*?)\)", 1)
                      ))

In [0]:
display(product_df)

product_name,product_id,category,ingestion_time,file_name,file_size,division,varient
SportsBar Energy Bar Choco Fudge (60g),25891101,Energy Bars,2026-09-05T06:35:08.056Z,products.csv,1388,Nutrition Bars,60g
SportsBar Energy Bar Choco Fudge (40g),25891102,Energy Bars,2026-09-05T06:35:08.056Z,products.csv,1388,Nutrition Bars,40g
SportsBar Energy Bar Choco Fudge (25g),25891103,Energy Bars,2026-09-05T06:35:08.056Z,products.csv,1388,Nutrition Bars,25g
SportsBar Protein Bar Peanut Crunch (45g),25891201,Protein Bars,2026-09-05T06:35:08.056Z,products.csv,1388,Nutrition Bars,45g
SportsBar Protein Bar Peanut Crunch (55g),25891202,Protein Bars,2026-09-05T06:35:08.056Z,products.csv,1388,Nutrition Bars,55g
SportsBar Protein Bar Peanut Crunch (65g),25891203,Protein Bars,2026-09-05T06:35:08.056Z,products.csv,1388,Nutrition Bars,65g
SportsBar Granola Crunch Honey Almond (400g),25891301,Granola & Cereals,2026-09-05T06:35:08.056Z,products.csv,1388,Breakfast Foods,400g
SportsBar Granola Crunch Honey Almond (300g),25891302,Granola & Cereals,2026-09-05T06:35:08.056Z,products.csv,1388,Breakfast Foods,300g
SportsBar Granola Crunch Honey Almond (200g),25891303,Granola & Cereals,2026-09-05T06:35:08.056Z,products.csv,1388,Breakfast Foods,200g
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,Recovery Dairy,2026-09-05T06:35:08.056Z,products.csv,1388,Dairy & Recovery,200g


In [0]:
product_df = (
    product_df.withColumn("product_code", (
        F.sha2(F.col('product_id').cast("string"), 256)
    ))
    .withColumn("product_id", (
        F.when(F.col("product_id").cast("string").rlike("^[0-9]+$") , F.col("product_id").cast("string"))
        .otherwise(F.lit(999999).cast("string")
        )
    ))
    .withColumn("product", F.col("product_name") )
)

In [0]:
product_df.show()

+--------------------+----------+-----------------+--------------------+------------+---------+--------------------+----------+--------------------+--------------------+
|        product_name|product_id|         category|      ingestion_time|   file_name|file_size|            division|   varient|        product_code|             product|
+--------------------+----------+-----------------+--------------------+------------+---------+--------------------+----------+--------------------+--------------------+
|SportsBar Energy ...|  25891101|      Energy Bars|2026-09-05 06:35:...|products.csv|     1388|      Nutrition Bars|       60g|521fcd441ab9d975c...|SportsBar Energy ...|
|SportsBar Energy ...|  25891102|      Energy Bars|2026-09-05 06:35:...|products.csv|     1388|      Nutrition Bars|       40g|7f6658d62c9204ef7...|SportsBar Energy ...|
|SportsBar Energy ...|  25891103|      Energy Bars|2026-09-05 06:35:...|products.csv|     1388|      Nutrition Bars|       25g|6c7000a2708d3a2ec...|Sp

In [0]:
product_df = product_df.select(
    "product_code"
    ,"division"
    ,"category"
    ,"product"
    ,"varient"
    ,"product_id"
    ,"ingestion_time"
    ,"file_name"
    ,"file_size"
)

In [0]:
display(product_df)

product_code,division,category,product,varient,product_id,ingestion_time,file_name,file_size
521fcd441ab9d975c4191fa2042c3824ca75763f1a81ed8f665e0b7d2e4c2913,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (60g),60g,25891101,2026-09-05T06:35:08.056Z,products.csv,1388
7f6658d62c9204ef7499c9dd87556e375e0edda7989f63c00e5fd3b147bc7612,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (40g),40g,25891102,2026-09-05T06:35:08.056Z,products.csv,1388
6c7000a2708d3a2ec0b87e57daed46779b14f86207cb9737b6c28ae1813aaed8,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (25g),25g,25891103,2026-09-05T06:35:08.056Z,products.csv,1388
d711f2934c5e710e59ea307db324072e1b5cf6315e641afb9be6b734dbb650eb,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (45g),45g,25891201,2026-09-05T06:35:08.056Z,products.csv,1388
a94fcc38f296de12368bef7dbd9cdaea162b2e986528582f5788d61966343cf4,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (55g),55g,25891202,2026-09-05T06:35:08.056Z,products.csv,1388
d84e9b44b1ae0668a15d77aec9b796391664bfb6245e56be926b34c1e5c72be1,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (65g),65g,25891203,2026-09-05T06:35:08.056Z,products.csv,1388
52d9158987029d4fa6688257dd034d2fee33778414ada3babe86f74e0a63dfc3,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (400g),400g,25891301,2026-09-05T06:35:08.056Z,products.csv,1388
399205b103ee9f68f358ee32b7fa9d5ec6ca2784eba72ab051191d8bd87d7a95,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (300g),300g,25891302,2026-09-05T06:35:08.056Z,products.csv,1388
11829ae9a1fba9dea2a44489773504db86d29211f09fa0e6f63bef020183cd0f,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (200g),200g,25891303,2026-09-05T06:35:08.056Z,products.csv,1388
c8eb3f3ffe65d3485867dc86e1df1a2fd9c177eafaacbec59e53802486abbe32,Dairy & Recovery,Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (200g),200g,25891401,2026-09-05T06:35:08.056Z,products.csv,1388


In [0]:
product_df.write\
.format("delta")\
.mode("overwrite")\
.option("mergeSchema", "true")\
.option("delta.enableChangeDataFeed", "true")\
.saveAsTable("business_to_business.silver.dim_sbproducts")


### Standardize the columns in gold layer according to Parent gold producst and merge the records in the parent's gold layer

In [0]:
product_gold = (
    spark.read.table("business_to_business.silver.dim_sbproducts")
)

In [0]:
display(product_gold)

product_code,division,category,product,varient,product_id,ingestion_time,file_name,file_size
e00d4df51d1a200e42b374b5821d6371fb36c1dd9cbbb480d581cf6da5b018f4,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (180g),180g,25891503,2026-09-05T06:35:08.056Z,products.csv,1388
7f6658d62c9204ef7499c9dd87556e375e0edda7989f63c00e5fd3b147bc7612,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (40g),40g,25891102,2026-09-05T06:35:08.056Z,products.csv,1388
399205b103ee9f68f358ee32b7fa9d5ec6ca2784eba72ab051191d8bd87d7a95,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (300g),300g,25891302,2026-09-05T06:35:08.056Z,products.csv,1388
53361f1d15f3967db2b9910b15dcb1e232f4bef98895594f616b4fde55548d39,Dairy & Recovery,Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (80g),80g,25891403,2026-09-05T06:35:08.056Z,products.csv,1388
52d9158987029d4fa6688257dd034d2fee33778414ada3babe86f74e0a63dfc3,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (400g),400g,25891301,2026-09-05T06:35:08.056Z,products.csv,1388
162576e78f6d2669a941bd670d03439db93822c13d6858eb09b181a97c8bf3eb,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (350g),350g,999999,2026-09-05T06:35:08.056Z,products.csv,1388
a94fcc38f296de12368bef7dbd9cdaea162b2e986528582f5788d61966343cf4,Nutrition Bars,Protein Bars,SportsBar Protein Bar Peanut Crunch (55g),55g,25891202,2026-09-05T06:35:08.056Z,products.csv,1388
38b61b697918c0ad776064b712efaf00297aaa6b5b2b453d8347226c5050f46e,Hydration & Electrolytes,Electrolyte Mix,SportsBar Electrolyte Mix Lemon-Lime (30 Sachets),30 Sachets,25891601,2026-09-05T06:35:08.056Z,products.csv,1388
b6b5f542b7bb3474f3e68bd1d13a7846b3bdba5e99a79e26ddd4e589c15944ac,Hydration & Electrolytes,Electrolyte Mix,SportsBar Electrolyte Mix Lemon-Lime (15 Sachets),15 Sachets,25891602,2026-09-05T06:35:08.056Z,products.csv,1388
11829ae9a1fba9dea2a44489773504db86d29211f09fa0e6f63bef020183cd0f,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (200g),200g,25891303,2026-09-05T06:35:08.056Z,products.csv,1388


In [0]:
product_gold = product_gold.select(
    "product_code"
    ,"division"
    ,"category"
    ,"product"
    ,"varient"
)

In [0]:
product_gold.write\
.format("delta")\
.mode("overwrite")\
.option("mergeSchema", "true")\
.option("delta.enableChangeDataFeed", "true")\
.saveAsTable("business_to_business.gold.dim_sbproducts")

### Merging Data Source with Parent

In [0]:
from delta.tables import DeltaTable
delta_table = DeltaTable.forName(spark, "business_to_business.gold.dim_products")
child_df = spark.read.table("business_to_business.gold.dim_sbproducts")
child_df.printSchema()

root
 |-- product_code: string (nullable = true)
 |-- division: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- varient: string (nullable = true)



In [0]:
delta_table.alias("target").merge(
    source=child_df.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "product_code": "source.product_code",
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.varient"
    }
).whenNotMatchedInsert(
    values = {
        "product_code": "source.product_code",
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.varient"
    }
).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]